# UPanda Tutorial: Planck PR3 (2018) Analysis with Low-ℓ Data

This tutorial demonstrates how to use UPanda to perform cosmological parameter inference using Planck 2018 CMB data (high-ℓ TT, TE, EE + low-ℓ TT, EE) with neural network emulators.

**Planck PR3** refers to the Planck 2018 data release, which includes:
- High-ℓ temperature and polarization power spectra (TT, TE, EE)
- Low-ℓ temperature and polarization power spectra (TT, EE)

## Table of Contents
1. [Setup and Installation](#setup)
2. [Loading Emulator Models](#loading-models)
3. [Configuring the MCMC Run](#configuration)
4. [Running MCMC Sampling](#running-mcmc)
5. [Visualizing Results with GetDist](#visualization)
6. [Understanding the Output](#understanding-output)


## 1. Setup and Installation {#setup}

First, let's set up UPanda and enable JAX for fast computations. JAX provides JIT compilation and automatic differentiation, making likelihood evaluations much faster.


In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import os

# Configure JAX for high precision (64-bit floating point)
from jax import config as jax_config
jax_config.update("jax_enable_x64", True)

# Enable JAX in UPanda
import UPanda.config as config
config.set_jax_enabled(True)

# Import UPanda modules
import UPanda.run_panda as run_panda
from UPanda.integrated_model_jax import IntegratedModel

print("✓ UPanda setup complete!")
print(f"✓ JAX enabled: {config.get_jax_enabled()}")


## 2. Loading Emulator Models {#loading-models}

For Planck CMB analysis, we need emulator models for three power spectra:
- **TT**: Temperature-temperature (T × T) power spectrum
- **TE**: Temperature-E-mode polarization (T × E) cross-power spectrum  
- **EE**: E-mode polarization (E × E) power spectrum

Each spectrum requires a separate emulator model. These models predict the CMB power spectra C_ℓ(ℓ) from cosmological parameters.

**Note**: The emulator models are stored in the `example_models/` directory.


In [ ]:
# Define paths to the emulator models
# Each model consists of a .pkl file (metadata) and .h5 file (neural network weights)
base_path = "../example_models"

# Load TT (temperature-temperature) emulator
tt_emu_model = IntegratedModel(None, None, None, None)
tt_emu_model.restore(f"{base_path}/tt_emu")
print("✓ TT emulator loaded")

# Load TE (temperature-polarization cross) emulator
te_emu_model = IntegratedModel(None, None, None, None)
te_emu_model.restore(f"{base_path}/te_emu")
print("✓ TE emulator loaded")

# Load EE (E-mode polarization) emulator
ee_emu_model = IntegratedModel(None, None, None, None)
ee_emu_model.restore(f"{base_path}/ee_emu")
print("✓ EE emulator loaded")

print(f"\n✓ All emulator models loaded successfully!")
print(f"  TT model parameters: {tt_emu_model.parameter_names[:5]}...")  # Show first 5


## 3. Configuring the MCMC Run {#configuration}

For Planck PR3 analysis, we use the `planck_2018_highell_lowell` likelihood, which combines:
- High-ℓ TT, TE, EE spectra (from Plik likelihood)
- Low-ℓ TT, EE spectra (from Commander and SimAll)

The `additional_args` dictionary allows us to control which data to include:
- `'add_lowl_tt': True` - Include low-ℓ TT data
- `'add_lowl_ee': True` - Include low-ℓ EE data (optional)

The configuration file defines the cosmological and nuisance parameters.


In [ ]:
# Set up the emulator model list
# For Planck, we need TT, TE, and EE models
# The outer list corresponds to experiments, inner list to models per experiment
total_emu_model_list = [[tt_emu_model, te_emu_model, ee_emu_model]]

# Specify which experiments to use
# planck_2018_highell_lowell combines high-ℓ and low-ℓ data
experiments_list = ["planck_2018_highell_lowell"]

# Compression vectors (empty for Planck - no compression needed)
compression_list = [[]]

# Path to configuration file
# This config defines priors for cosmological and nuisance parameters
config_file = "../configs/config_example_cmb_planck_pr4_lowl.yaml"

# Additional arguments for the likelihood
# These control which data components to include
additional_args = {
    'add_lowl_tt': True,  # Include low-ℓ TT data (Commander)
    'integrated_model': True  # Use integrated model format
}

print("✓ Configuration set up!")
print(f"  Experiments: {experiments_list}")
print(f"  Config file: {config_file}")
print(f"  Additional args: {additional_args}")


## 4. Running MCMC Sampling {#running-mcmc}

Now we'll run the MCMC sampler. For Planck analysis with multiple spectra and nuisance parameters, convergence typically requires more samples than BAO analysis.

**Note**: Planck likelihoods include many nuisance parameters (calibration, foregrounds, etc.), so the parameter space is higher-dimensional. Use `n_samples >= 1e6` and `n_walkers >= 128` for production runs.


In [ ]:
# Run MCMC sampling
# For tutorial purposes, we use fewer samples - increase for production runs
print("Starting MCMC sampling for Planck PR3...")
print("This may take 10-30 minutes depending on your hardware.")
print("Planck analysis includes many nuisance parameters, so it's more computationally intensive.")

params_chain, loglike = run_panda.run_mcmc(
    total_emu_model_list=total_emu_model_list,
    experiments_list=experiments_list,
    compression_list=compression_list,
    filename_config=config_file,
    n_samples=100000,  # Use 1e6 or more for production
    n_walkers=64,       # Use 128 or more for production
    sampler="emcee",    # Options: "emcee", "nautilus", "ultranest", "hmc"
    additional_args=additional_args
)

print("\n✓ MCMC sampling complete!")
print(f"  Chain shape: {params_chain.shape}")
print(f"  Parameter names: {params_chain.dtype.names}")
print(f"  Number of parameters: {len(params_chain.dtype.names)}")
print(f"  Log-likelihood range: [{np.min(loglike):.2f}, {np.max(loglike):.2f}]")


## 5. Visualizing Results with GetDist {#visualization}

GetDist is a powerful tool for analyzing and visualizing MCMC chains. For Planck analysis, we'll create plots focusing on cosmological parameters and optionally nuisance parameters.


In [ ]:
# Convert chain to GetDist format
import getdist
from getdist import MCSamples, plots

# Extract parameter names and values
param_names = list(params_chain.dtype.names)
chain_array = np.column_stack([params_chain[name] for name in param_names])

# Create GetDist MCSamples object
samples = MCSamples(
    samples=chain_array,
    names=param_names,
    labels=[name.replace('_', ' ').title() for name in param_names],
    label="Planck PR3"
)

print("✓ GetDist samples created!")
print(f"  Number of samples: {len(samples.samples)}")
print(f"  Number of parameters: {samples.num_vars}")


In [ ]:
# Create corner plot for cosmological parameters only
# Planck analysis includes many nuisance parameters, so we focus on cosmology first
cosmo_params = ["omega_b", "omega_cdm", "n_s", "tau", "h", "sigma8"]

g = plots.get_subplot_plotter()
g.triangle_plot([samples], params=cosmo_params, filled=True, contour_colors=['steelblue'])
plt.show()


In [ ]:
# Print summary statistics for cosmological parameters
print("\n" + "="*70)
print("Cosmological Parameter Constraints (Mean ± Std, 68% CI)")
print("="*70)
for name in cosmo_params:
    if name in param_names:
        mean = np.mean(params_chain[name])
        std = np.std(params_chain[name])
        lower = np.percentile(params_chain[name], 16)
        upper = np.percentile(params_chain[name], 84)
        print(f"{name:15s}: {mean:8.5f} ± {std:8.5f}  [{lower:8.5f}, {upper:8.5f}]")
print("="*70)


In [ ]:
# Example: Check convergence by plotting log-likelihood evolution
plt.figure(figsize=(12, 4))
plt.plot(loglike, alpha=0.5, linewidth=0.5)
plt.xlabel('Sample Number', fontsize=12)
plt.ylabel('Log-Likelihood', fontsize=12)
plt.title('Log-Likelihood Evolution During MCMC (Planck PR3)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Check convergence statistics
mean_loglike = np.mean(loglike[-20000:])  # Last 20k samples
std_loglike = np.std(loglike[-20000:])
print(f"\nLog-likelihood statistics (last 20k samples):")
print(f"  Mean: {mean_loglike:.2f}")
print(f"  Std:  {std_loglike:.2f}")
if std_loglike < 10:
    print("  ✓ Chain appears converged")
else:
    print("  ⚠ Chain may need more samples")


In [ ]:
# Example: Compute derived parameter omega_m
omega_m = (params_chain['omega_cdm'] + params_chain['omega_b']) / (params_chain['h']**2)

# Compute statistics
mean_om = np.mean(omega_m)
std_om = np.std(omega_m)
lower_om = np.percentile(omega_m, 16)
upper_om = np.percentile(omega_m, 84)

print("Derived Parameter: Omega_m")
print(f"  Mean: {mean_om:.5f}")
print(f"  Std:  {std_om:.5f}")
print(f"  68% CI: [{lower_om:.5f}, {upper_om:.5f}]")

# Plot comparison of key cosmological parameters
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

key_params = ['omega_b', 'omega_cdm', 'h', 'n_s', 'tau', 'sigma8']
for i, param in enumerate(key_params):
    if param in param_names:
        axes[i].hist(params_chain[param], bins=50, density=True, alpha=0.7, 
                    color='steelblue', edgecolor='black')
        mean_val = np.mean(params_chain[param])
        axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=2)
        axes[i].set_xlabel(param.replace('_', ' ').title(), fontsize=12)
        axes[i].set_ylabel('Probability Density', fontsize=10)
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
